# Mitigating QAOA errors for MaxCut with EMRG

The Quantum Approximate Optimization Algorithm (QAOA) finds approximate solutions to
combinatorial optimization problems by alternating cost and mixer layers on a quantum
circuit. MaxCut -- partitioning a graph's vertices to maximize the number of edges
between partitions -- is the standard benchmark.

QAOA circuits are typically deeper than VQE ansatze for small molecules because each
edge in the graph contributes an RZZ gate to the cost layer. On noisy hardware, this
depth degrades the quality of the approximate solution. Error mitigation can partially
recover the lost signal.

This notebook builds a 2-layer QAOA circuit for a 5-node MaxCut instance, runs it
through EMRG, and compares the ZNE and PEC recipes. The goal is to show how EMRG
adapts its recommendation to a deeper, noisier circuit.

## Setup

In [ ]:
# pip install emrg qiskit

In [ ]:
import numpy as np
from qiskit import QuantumCircuit

import emrg

## Define the MaxCut graph

A 5-node graph with 6 edges. Each edge becomes an RZZ gate in the QAOA cost layer.

In [ ]:
n_nodes = 5
edges = [
    (0, 1),
    (0, 2),
    (1, 2),
    (1, 3),
    (2, 4),
    (3, 4),
]

print(f"Nodes: {n_nodes}")
print(f"Edges: {len(edges)}")
print(f"Edge list: {edges}")

## Build the QAOA circuit

Each QAOA layer consists of:
1. **Cost layer**: an RZZ gate for each edge, encoding the MaxCut objective
2. **Mixer layer**: an RX gate on each qubit, providing exploration

We use 2 QAOA layers (p=2). The parameters are bound to fixed values for this
demonstration.

In [ ]:
n_layers = 2

qc = QuantumCircuit(n_nodes, n_nodes)

# Initial superposition
for q in range(n_nodes):
    qc.h(q)

for layer in range(n_layers):
    # Cost layer: RZZ for each edge
    gamma = np.random.uniform(0, 2 * np.pi)
    for i, j in edges:
        qc.rzz(gamma, i, j)

    # Mixer layer: RX on each qubit
    beta = np.random.uniform(0, 2 * np.pi)
    for q in range(n_nodes):
        qc.rx(beta, q)

qc.measure(range(n_nodes), range(n_nodes))

print(f"Qubits: {qc.num_qubits}")
print(f"Depth:  {qc.depth()}")
print(qc.draw())

This circuit is deeper than the H₂ VQE ansatz from the companion notebook. More
multi-qubit gates (12 RZZ gates across 2 layers) and higher depth will influence
EMRG's technique selection.

## Analyze the circuit

In [ ]:
features = emrg.analyze_circuit(qc)

print(f"Qubits:              {features.num_qubits}")
print(f"Depth:               {features.depth}")
print(f"Total gates:         {features.total_gate_count}")
print(f"Multi-qubit gates:   {features.multi_qubit_gate_count}")
print(f"Single-qubit gates:  {features.single_qubit_gate_count}")
print(f"Noise estimate:      {features.estimated_noise_factor}")
print(f"Noise category:      {features.noise_category}")
print(f"PEC overhead est:    {features.pec_overhead_estimate:.2f}")
print(f"Layer heterogeneity: {features.layer_heterogeneity:.4f}")

Compared to the H₂ circuit:
- More multi-qubit gates (12 RZZ vs 6 CNOT)
- Higher noise estimate
- Higher PEC overhead -- PEC may become impractical

These differences will cause EMRG to select a different ZNE configuration.

## Generate the mitigation recipe

In [ ]:
result = emrg.generate_recipe(qc, explain=True)

print(f"Technique: {result.recipe.technique}")
print(f"Factory:   {result.recipe.factory_name}")
print(f"Scaling:   {result.recipe.scaling_method}")
print(f"Scale factors: {result.recipe.scale_factors}")
print()
print(result.code)

The recipe reflects the circuit's characteristics. A deeper QAOA circuit with more
multi-qubit gates receives a different factory and/or more scale factors than the
shallow H₂ circuit. The rationale lines explain the specific reasoning.

In [ ]:
for line in result.rationale:
    print(f"  - {line}")

## Technique comparison: ZNE vs PEC

Force both techniques and compare.

In [ ]:
zne_result = emrg.generate_recipe(qc, technique="zne", explain=True)
pec_result = emrg.generate_recipe(
    qc, technique="pec", noise_model_available=True, explain=True
)

print("=== ZNE ===")
print(f"Factory:   {zne_result.recipe.factory_name}")
print(f"Scaling:   {zne_result.recipe.scaling_method}")
print(f"Overhead:  {zne_result.recipe.estimated_overhead:.0f}x")
print()
print("=== PEC ===")
print(f"Samples:   {pec_result.recipe.factory_kwargs.get('num_samples', 'N/A')}")
print(f"Overhead:  {pec_result.recipe.estimated_overhead:.1f}x")

For a circuit with 12 multi-qubit gates, PEC overhead grows exponentially. The
overhead estimate (`exp(2 * noise_factor * multi_qubit_gates)`) is significantly
higher than for the H₂ circuit. In practice, PEC at this scale requires many more
samples to converge, which may exceed practical shot budgets.

ZNE remains viable regardless of circuit size. The tradeoff is that ZNE is biased
(it relies on extrapolation), while PEC is unbiased when the noise model is accurate.

## Summary

Different circuits produce different EMRG recommendations. The H₂ VQE ansatz (shallow,
few multi-qubit gates) gets a simple ZNE configuration and is a good PEC candidate.
The QAOA MaxCut circuit (deeper, many RZZ gates) gets a more aggressive ZNE
configuration and pushes PEC toward impractical overhead.

This is the point of EMRG: automatic, circuit-aware technique selection.

Links:
- [EMRG on GitHub](https://github.com/FedorShind/EMRG)
- [EMRG on PyPI](https://pypi.org/project/emrg/)
- [Mitiq documentation](https://mitiq.readthedocs.io/)